In [ ]:
# YOLOv8 Object Detection - Betel Insects Dataset
# Classes: Green-Leafhopper, aphid, beetle, grasshopper
# Features: Training, Evaluation, OOD Detection (Mahalanobis Distance)
# Platform: Kaggle Notebook
# =============================================================================

# ─────────────────────────────────────────────
# CELL 1: Install Dependencies
# ─────────────────────────────────────────────
!pip install ultralytics onnx onnxruntime onnxscript -q
import os, json, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image
import yaml
import cv2
import torch
from collections import defaultdict

from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.covariance import EmpiricalCovariance

warnings.filterwarnings("ignore")

In [ ]:
# ─────────────────────────────────────────────
# CELL 2: Configuration
# ─────────────────────────────────────────────

# Dataset & output paths
DATASET_ROOT   = Path("/kaggle/input/datasets/amandapriyashantha/betel-pets-obj-01")
WORKING_DIR    = Path("/kaggle/working")
FILTERED_DIR   = WORKING_DIR / "betel_filtered"
OUTPUT_DIR     = WORKING_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Classes to KEEP (indices in the ORIGINAL dataset must be discovered first)
KEEP_CLASSES   = ["Green-Leafhopper", "aphid", "beetle", "grasshopper"]
REMOVE_CLASSES = ["armyworm", "sawfly"]

# Training hyper-parameters
IMG_SIZE  = 640
EPOCHS    = 25
BATCH     = 16
DEVICE    = 0 if torch.cuda.is_available() else "cpu"
MODEL_CFG = "yolov8m.pt"           # medium backbone – good balance for Kaggle P100

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"Device         : {DEVICE}")
print(f"Dataset root   : {DATASET_ROOT}")


In [ ]:
# ─────────────────────────────────────────────
# CELL 3: Discover Original Class Mapping
# ─────────────────────────────────────────────

orig_yaml_path = DATASET_ROOT / "data.yaml"
with open(orig_yaml_path) as f:
    orig_cfg = yaml.safe_load(f)

orig_classes = orig_cfg["names"]           # list of class names
print("Original classes:", orig_classes)

# Map: original index → class name
orig_id2name = {i: n for i, n in enumerate(orig_classes)}
orig_name2id = {n: i for i, n in enumerate(orig_classes)}

# New consecutive mapping (keep order)
new_classes  = [c for c in orig_classes if c in KEEP_CLASSES]
new_name2id  = {n: i for i, n in enumerate(new_classes)}
new_id2name  = {i: n for i, n in enumerate(new_classes)}

# Set of original IDs to KEEP
keep_orig_ids = {orig_name2id[c] for c in KEEP_CLASSES if c in orig_name2id}
# Remap: original_id → new_id
remap = {oid: new_name2id[orig_id2name[oid]] for oid in keep_orig_ids}

print("New classes    :", new_classes)
print("Remap table    :", remap)

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 (UPDATED): Build dataset from train folder only
#   - Cap 2400 images per class
#   - Split → 80% train / 15% val / 5% test
# ─────────────────────────────────────────────
 
import shutil, random
from collections import Counter, defaultdict
from pathlib import Path
 
MAX_PER_CLASS  = 2400
TRAIN_RATIO    = 0.80
VAL_RATIO      = 0.15
TEST_RATIO     = 0.05   # remainder after train+val
random.seed(42)
 
# Clean and recreate output split dirs
for split in ["train", "valid", "test"]:
    for sub in ["images", "labels"]:
        d = FILTERED_DIR / split / sub
        if d.exists():
            shutil.rmtree(d)
        d.mkdir(parents=True, exist_ok=True)
 
 
# ── Step 1: Scan ONLY the original train folder ──────────────────────
src_img_dir = DATASET_ROOT / "train" / "images"
src_lbl_dir = DATASET_ROOT / "train" / "labels"
 
candidates = []   # (img_path, lbl_path, primary_new_class_id, all_new_class_ids)
 
for img_path in sorted(src_img_dir.iterdir()):
    if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp"}:
        continue
    lbl_path = src_lbl_dir / (img_path.stem + ".txt")
    if not lbl_path.exists():
        continue
 
    new_ids = []
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                orig_cid = int(parts[0])
                if orig_cid in keep_orig_ids:
                    new_ids.append(remap[orig_cid])
 
    if not new_ids:
        continue   # image has no kept-class annotations
 
    primary = Counter(new_ids).most_common(1)[0][0]
    candidates.append((img_path, lbl_path, primary, new_ids))
 
print(f"Total candidate images (all classes): {len(candidates)}")
 
# Raw annotation counts before capping
raw_ann = defaultdict(int)
for _, _, _, ids in candidates:
    for i in ids:
        raw_ann[i] += 1
print("Raw annotation counts per class:")
for cid in sorted(raw_ann):
    print(f"  {new_id2name[cid]:20s}: {raw_ann[cid]} annotations")
 
 
# ── Step 2: Cap at MAX_PER_CLASS per primary class ───────────────────
random.shuffle(candidates)
 
class_counts = defaultdict(int)
capped = []
for item in candidates:
    _, _, primary, _ = item
    if class_counts[primary] < MAX_PER_CLASS:
        capped.append(item)
        class_counts[primary] += 1
 
print(f"\nAfter capping at {MAX_PER_CLASS}/class:")
for cid in sorted(class_counts):
    print(f"  {new_id2name[cid]:20s}: {class_counts[cid]} images")
print(f"  Total selected         : {len(capped)} images")
 
 
# ── Step 3: Stratified split per class → train / val / test ──────────
# Group by primary class, split each group, then merge
 
train_items, val_items, test_items = [], [], []
 
for cid in sorted(class_counts):
    class_pool = [item for item in capped if item[2] == cid]
    random.shuffle(class_pool)
    n          = len(class_pool)
    n_val      = round(n * VAL_RATIO)
    n_test     = round(n * TEST_RATIO)
    n_train    = n - n_val - n_test
 
    train_items.extend(class_pool[:n_train])
    val_items.extend(class_pool[n_train : n_train + n_val])
    test_items.extend(class_pool[n_train + n_val :])
 
    print(f"  {new_id2name[cid]:20s}  "
          f"train={n_train}  val={n_val}  test={n_test}")
 
print(f"\nFinal split totals → "
      f"train={len(train_items)}  val={len(val_items)}  test={len(test_items)}")
 
 
# ── Step 4: Helper — write remapped label and copy image ─────────────
def write_split_item(img_path: Path, lbl_path: Path, split: str):
    """Remap label file and copy image into the filtered split folder."""
    dst_lbl = FILTERED_DIR / split / "labels" / (img_path.stem + ".txt")
    dst_img = FILTERED_DIR / split / "images" / img_path.name
 
    lines_out = []
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            orig_cid = int(parts[0])
            if orig_cid in keep_orig_ids:
                new_id = remap[orig_cid]
                lines_out.append(f"{new_id} " + " ".join(parts[1:]))
 
    if not lines_out:
        return False
 
    with open(dst_lbl, "w") as f:
        f.write("\n".join(lines_out) + "\n")
    shutil.copy2(img_path, dst_img)
    return True
 
 
# ── Step 5: Write all splits ─────────────────────────────────────────
for split, items in [("train", train_items),
                     ("valid",  val_items),
                     ("test",   test_items)]:
    written = 0
    for img_path, lbl_path, _, _ in items:
        if write_split_item(img_path, lbl_path, split):
            written += 1
    print(f"  [{split}] wrote {written} images")
 
# ── Step 6: Annotation counts per split (verification) ───────────────
print("\nAnnotation counts per class per split:")
for split in ["train", "valid", "test"]:
    lbl_dir = FILTERED_DIR / split / "labels"
    ann_counts = defaultdict(int)
    for lbl_file in lbl_dir.glob("*.txt"):
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    ann_counts[int(parts[0])] += 1
    print(f"  [{split}]  " +
          "  ".join(f"{new_id2name[c]}={ann_counts[c]}"
                    for c in sorted(ann_counts)))
 
print("\nDone.")


In [ ]:
# ─────────────────────────────────────────────
# CELL 5 (UNCHANGED): Write filtered data.yaml
# ─────────────────────────────────────────────
 
filtered_yaml = {
    "path"  : str(FILTERED_DIR),
    "train" : "train/images",
    "val"   : "valid/images",
    "test"  : "test/images",
    "nc"    : len(new_classes),
    "names" : new_classes,
}
 
yaml_path = FILTERED_DIR / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(filtered_yaml, f, default_flow_style=False)
 
print("Filtered data.yaml written:")
print(open(yaml_path).read())

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 (UPDATED): Train YOLOv8 with minimal augmentation
# ─────────────────────────────────────────────
 
model = YOLO(MODEL_CFG)
 
results = model.train(
    data         = str(yaml_path),
    epochs       = 30,            # bumped from 25 — model hadn't converged
    imgsz        = IMG_SIZE,
    batch        = BATCH,
    device       = DEVICE,
    project      = str(WORKING_DIR / "runs"),
    name         = "betel_insects",
    exist_ok     = True,
 
    # ── Minimal augmentation (Fix 2) ────────────────────────────────
    # Geometric — safe for insect detection
    fliplr       = 0.5,     # horizontal flip
    flipud       = 0.0,     # vertical flip — insects don't appear upside-down
    degrees      = 5.0,     # slight rotation (±5°)
    translate    = 0.1,     # small translation
    scale        = 0.3,     # zoom in/out
    shear        = 0.0,
    perspective  = 0.0,
 
    # Colour — handles lighting/camera variation
    hsv_h        = 0.015,   # hue jitter
    hsv_s        = 0.4,     # saturation jitter
    hsv_v        = 0.4,     # brightness jitter
 
    # Keep everything else OFF
    mosaic       = 0.0,
    mixup        = 0.0,
    copy_paste   = 0.0,
    augment      = False,   # disables built-in test-time augment (inference)
 
    plots        = True,
    save         = True,
    verbose      = True,
    patience     = 20,      # early-stop if no improvement for 20 epochs
)
 
TRAIN_DIR = Path(results.save_dir)
BEST_PT   = TRAIN_DIR / "weights" / "best.pt"
print(f"\nBest weights: {BEST_PT}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 7: Plot Training Curves (Accuracy & Loss)
# ─────────────────────────────────────────────

results_csv = TRAIN_DIR / "results.csv"
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

print("Available columns:", df.columns.tolist())

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("YOLOv8 Training Curves – Betel Insects", fontsize=16, fontweight="bold")

epochs_range = range(1, len(df) + 1)

# ── Loss curves ──────────────────────────────
loss_cols = {
    "Box Loss (train)"  : ("train/box_loss",  "steelblue"),
    "Cls Loss (train)"  : ("train/cls_loss",  "tomato"),
    "DFL Loss (train)"  : ("train/dfl_loss",  "seagreen"),
    "Box Loss (val)"    : ("val/box_loss",    "steelblue"),
    "Cls Loss (val)"    : ("val/cls_loss",    "tomato"),
    "DFL Loss (val)"    : ("val/dfl_loss",    "seagreen"),
}

for ax, (title, (col, color)) in zip(axes[0], [
        ("Train Box Loss", ("train/box_loss",  "steelblue")),
        ("Train Cls Loss", ("train/cls_loss",  "tomato")),
        ("Train DFL Loss", ("train/dfl_loss",  "seagreen")),
]):
    if col in df.columns:
        ax.plot(epochs_range, df[col], color=color, linewidth=2)
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.grid(alpha=0.3)

for ax, (title, (col, color)) in zip(axes[1], [
        ("Val Box Loss",   ("val/box_loss",    "steelblue")),
        ("Val Cls Loss",   ("val/cls_loss",    "tomato")),
        ("Val DFL Loss",   ("val/dfl_loss",    "seagreen")),
]):
    if col in df.columns:
        ax.plot(epochs_range, df[col], color=color, linewidth=2, linestyle="--")
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.grid(alpha=0.3)

plt.tight_layout()
loss_fig_path = OUTPUT_DIR / "loss_curves.png"
plt.savefig(loss_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {loss_fig_path}")

# ── Accuracy / mAP curves ────────────────────
acc_cols = [
    ("metrics/precision(B)", "Precision",  "dodgerblue"),
    ("metrics/recall(B)",    "Recall",     "darkorange"),
    ("metrics/mAP50(B)",     "mAP@0.50",   "mediumseagreen"),
    ("metrics/mAP50-95(B)",  "mAP@.5:.95", "mediumpurple"),
]

fig2, ax2 = plt.subplots(figsize=(10, 5))
for col, label, color in acc_cols:
    if col in df.columns:
        ax2.plot(epochs_range, df[col], label=label, color=color, linewidth=2)
ax2.set_title("Validation Metrics per Epoch", fontsize=14, fontweight="bold")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Score")
ax2.legend(); ax2.grid(alpha=0.3)
acc_fig_path = OUTPUT_DIR / "accuracy_curves.png"
plt.savefig(acc_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {acc_fig_path}")

# ─────────────────────────────────────────────
# CELL 8 (UPDATED): Evaluate on Test Set with IoU-based Hungarian Matching
# ─────────────────────────────────────────────
 
from scipy.optimize import linear_sum_assignment
 
best_model = YOLO(str(BEST_PT))
 
IOU_MATCH_THRESHOLD = 0.5   # GT↔Pred match requires IoU ≥ 0.5
 
 
def box_iou(boxA, boxB):
    """Compute IoU between two [x1,y1,x2,y2] boxes."""
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    interW = max(0.0, xB - xA)
    interH = max(0.0, yB - yA)
    inter  = interW * interH
    areaA  = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB  = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    union  = areaA + areaB - inter
    return inter / union if union > 0 else 0.0
 
 
def yolo_to_xyxy(cx, cy, bw, bh, img_w, img_h):
    """Convert YOLO normalised [cx,cy,w,h] → pixel [x1,y1,x2,y2]."""
    x1 = (cx - bw / 2) * img_w
    y1 = (cy - bh / 2) * img_h
    x2 = (cx + bw / 2) * img_w
    y2 = (cy + bh / 2) * img_h
    return [x1, y1, x2, y2]
 
 
y_true, y_pred = [], []
 
test_img_dir = FILTERED_DIR / "test" / "images"
test_lbl_dir = FILTERED_DIR / "test" / "labels"
 
for img_path in sorted(test_img_dir.iterdir()):
    lbl_path = test_lbl_dir / (img_path.stem + ".txt")
    if not lbl_path.exists():
        continue
 
    # ── Load image dimensions for coordinate conversion ─────────────
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        continue
    img_h, img_w = img_bgr.shape[:2]
 
    # ── Parse ground-truth boxes ─────────────────────────────────────
    gt_boxes, gt_ids = [], []
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cid = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:5])
            gt_boxes.append(yolo_to_xyxy(cx, cy, bw, bh, img_w, img_h))
            gt_ids.append(cid)
 
    if not gt_ids:
        continue

In [ ]:
# ── Run YOLOv8 prediction ─────────────────────────────────────────
    preds     = best_model.predict(str(img_path), conf=0.25, iou=0.45,
                                   verbose=False, device=DEVICE)[0]
    pred_boxes = (preds.boxes.xyxy.cpu().numpy().tolist()
                  if preds.boxes and len(preds.boxes) else [])
    pred_ids   = (preds.boxes.cls.cpu().numpy().astype(int).tolist()
                  if preds.boxes and len(preds.boxes) else [])

In [ ]:
# ── No predictions → all GT objects are missed ───────────────────
    if not pred_ids:
        y_true.extend(gt_ids)
        y_pred.extend([-1] * len(gt_ids))
        continue

In [ ]:
# ── Build IoU cost matrix (GT rows × Pred cols) ──────────────────
    n_gt, n_pred = len(gt_ids), len(pred_ids)
    cost = np.zeros((n_gt, n_pred), dtype=np.float32)
    for i, gb in enumerate(gt_boxes):
        for j, pb in enumerate(pred_boxes):
            cost[i, j] = 1.0 - box_iou(gb, pb)   # minimise cost = maximise IoU

In [ ]:
  # ── Hungarian optimal assignment ──────────────────────────────────
    row_ind, col_ind = linear_sum_assignment(cost)
 
    matched_gt_set = set()
    for r, c in zip(row_ind, col_ind):
        achieved_iou = 1.0 - cost[r, c]
        if achieved_iou >= IOU_MATCH_THRESHOLD:
            # Correctly matched GT ↔ Pred pair
            y_true.append(gt_ids[r])
            y_pred.append(pred_ids[c])
            matched_gt_set.add(r)
        # else: low-IoU assignment → treat GT as missed (handled below)
 
    # ── Unmatched GT objects = missed detections ──────────────────────
    for i in range(n_gt):
        if i not in matched_gt_set:
            y_true.append(gt_ids[i])
            y_pred.append(-1)   # -1 = "missed"

In [ ]:
# ── Classification Report ──────────────────────────────────────────────
report_labels = list(range(len(new_classes)))
report_names  = new_classes
 
print("\n" + "=" * 60)
print("CLASSIFICATION REPORT  (IoU-matched, threshold=0.50)")
print("=" * 60)
report_str = classification_report(
    y_true, y_pred,
    labels       = report_labels,
    target_names = report_names,
    zero_division= 0,
)
print(report_str)
 
report_path = OUTPUT_DIR / "classification_report.txt"
with open(report_path, "w") as f:
    f.write("YOLOv8 Betel Insects – Classification Report\n")
    f.write("(GT↔Pred matched via Hungarian algorithm, IoU≥0.50)\n")
    f.write("=" * 60 + "\n")
    f.write(report_str)
print(f"Saved: {report_path}")

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────────────────
# Build label set: known classes + "missed" for undetected GT objects
cm_labels     = sorted(set(y_true + y_pred))
disp_label_names = [new_id2name.get(i, "missed") for i in cm_labels]
 
cm      = confusion_matrix(y_true, y_pred, labels=cm_labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)
 
fig3, axes3 = plt.subplots(1, 2, figsize=(16, 6))
fig3.suptitle("Confusion Matrix – Betel Insects Test Set\n"
              "(IoU≥0.50 Hungarian matching)",
              fontsize=14, fontweight="bold")
 
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=disp_label_names, yticklabels=disp_label_names,
            ax=axes3[0], linewidths=0.5)
axes3[0].set_title("Raw Counts")
axes3[0].set_xlabel("Predicted"); axes3[0].set_ylabel("True")
 
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=disp_label_names, yticklabels=disp_label_names,
            ax=axes3[1], linewidths=0.5, vmin=0, vmax=1)
axes3[1].set_title("Normalised")
axes3[1].set_xlabel("Predicted"); axes3[1].set_ylabel("True")
 
plt.tight_layout()
cm_path = OUTPUT_DIR / "confusion_matrix.png"
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {cm_path}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 9: Export model to ONNX
# ─────────────────────────────────────────────

onnx_path = OUTPUT_DIR / "model.onnx"
best_model.export(format="onnx", imgsz=IMG_SIZE, opset=12, simplify=True)
exported = BEST_PT.parent / (BEST_PT.stem + ".onnx")
shutil.copy2(exported, onnx_path)
print(f"Saved: {onnx_path}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 10: Build Feature Extractor (backbone only → ONNX)
# ─────────────────────────────────────────────
"""
We hook into the YOLOv8 backbone to extract 1280-d feature vectors.
These vectors are later used for Mahalanobis OOD detection.
"""

import onnx
import onnxruntime as ort
import torch.nn as nn

class BackboneWrapper(nn.Module):
    """Wraps YOLO model to return flattened backbone features (after SPPF)."""
    def __init__(self, yolo_model):
        super().__init__()
        self.backbone = yolo_model.model.model[:10]   # layers 0-9 = backbone + SPPF

    def forward(self, x):
        for layer in self.backbone:
            x = layer(x)
        return x.flatten(1)   # (B, C*H*W) → use adaptive pool first
    
class BackboneWithPool(nn.Module):
    def __init__(self, yolo_model):
        super().__init__()
        self.backbone = yolo_model.model.model[:10]
        self.pool     = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        for layer in self.backbone:
            x = layer(x)
        x = self.pool(x)
        return x.flatten(1)   # (B, C)

feat_model = BackboneWithPool(best_model).eval()
if torch.cuda.is_available():
    feat_model = feat_model.cuda()

dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
if torch.cuda.is_available():
    dummy = dummy.cuda()

feat_onnx_path = OUTPUT_DIR / "feature_extractor.onnx"
torch.onnx.export(
    feat_model, dummy,
    str(feat_onnx_path),
    input_names  = ["images"],
    output_names = ["features"],
    dynamic_axes = {"images": {0: "batch"}, "features": {0: "batch"}},
    opset_version= 12,
)
print(f"Saved: {feat_onnx_path}")


In [ ]:
# Quick sanity check
sess = ort.InferenceSession(str(feat_onnx_path),
                            providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
feat_dim = sess.run(None, {"images": np.zeros((1, 3, IMG_SIZE, IMG_SIZE), dtype=np.float32)})[0].shape[1]
print(f"Feature dimension: {feat_dim}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 11: Fit Mahalanobis OOD Detector on Training Features
# ─────────────────────────────────────────────

def extract_features_batch(img_dir: Path, session: ort.InferenceSession,
                            img_size: int = 640, max_imgs: int = 2000):
    """
    Extract backbone features for all images in img_dir.
    Returns array of shape (N, feat_dim).
    """
    feats, paths = [], []
    img_paths = sorted(img_dir.glob("*.*"))[:max_imgs]
    for ip in img_paths:
        try:
            img = cv2.imread(str(ip))
            if img is None: continue
            img = cv2.resize(img, (img_size, img_size))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            tensor = img.astype(np.float32) / 255.0
            tensor = np.transpose(tensor, (2, 0, 1))[None]   # (1,3,H,W)
            feat   = session.run(None, {"images": tensor})[0] # (1, feat_dim)
            feats.append(feat[0])
            paths.append(str(ip))
        except Exception as e:
            print(f"  Skipping {ip.name}: {e}")
    return np.array(feats), paths


print("Extracting training features …")
feat_sess = ort.InferenceSession(str(feat_onnx_path),
                                  providers=["CUDAExecutionProvider","CPUExecutionProvider"])
# AFTER — pass max_imgs large enough to cover full train set
train_feats, train_paths = extract_features_batch(
    FILTERED_DIR / "train" / "images", feat_sess, IMG_SIZE, max_imgs=8000)
print(f"  Extracted {len(train_feats)} feature vectors, dim={train_feats.shape[1]}")

# Per-class feature banks
train_lbl_dir = FILTERED_DIR / "train" / "labels"
class_feats   = defaultdict(list)
for feat, fp in zip(train_feats, train_paths):
    stem     = Path(fp).stem
    lbl_file = train_lbl_dir / (stem + ".txt")
    if not lbl_file.exists(): continue
    with open(lbl_file) as f:
        ids = [int(l.split()[0]) for l in f if l.strip()]
    for cid in set(ids):
        class_feats[cid].append(feat)

# Fit per-class Gaussian (mean + covariance)
class_means = {}
class_covs  = {}
EPSILON     = 1e-6   # regularisation

for cid, vecs in class_feats.items():
    arr = np.array(vecs)
    mu  = arr.mean(axis=0)
    cov = EmpiricalCovariance().fit(arr)
    class_means[cid] = mu
    class_covs[cid]  = cov.precision_   # inverse covariance

# Global mean/cov (class-agnostic OOD)
global_mu  = train_feats.mean(axis=0)
global_cov = EmpiricalCovariance().fit(train_feats)
global_prec = global_cov.precision_

print("Fitted Mahalanobis parameters for classes:",
      [new_id2name[c] for c in sorted(class_means)])

In [ ]:
# ─────────────────────────────────────────────
# CELL 12: Determine OOD Threshold on Validation Set
# ─────────────────────────────────────────────

def mahalanobis_score(feat: np.ndarray, mu: np.ndarray, prec: np.ndarray) -> float:
    """Scalar Mahalanobis distance for a single feature vector."""
    diff = feat - mu
    return float(np.sqrt(diff @ prec @ diff))


def min_class_mahalanobis(feat: np.ndarray) -> float:
    """Minimum Mahalanobis distance across all known classes."""
    scores = [mahalanobis_score(feat, class_means[c], class_covs[c])
              for c in class_means]
    return min(scores)


print("Extracting validation features …")
val_feats, val_paths = extract_features_batch(
    FILTERED_DIR / "valid" / "images", feat_sess, IMG_SIZE, max_imgs=2000)
val_scores = np.array([min_class_mahalanobis(f) for f in val_feats])

# Threshold = mean + 3σ on in-distribution (val) scores
ood_threshold = float(val_scores.mean() + 3 * val_scores.std())
print(f"  Val scores  → mean={val_scores.mean():.2f}  std={val_scores.std():.2f}")
print(f"  OOD threshold (μ+3σ): {ood_threshold:.4f}")

# Plot score distribution
fig4, ax4 = plt.subplots(figsize=(9, 4))
ax4.hist(val_scores, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
ax4.axvline(ood_threshold, color="red", linestyle="--", linewidth=2,
            label=f"Threshold = {ood_threshold:.2f}")
ax4.set_title("Mahalanobis Score Distribution (Validation – In-Distribution)")
ax4.set_xlabel("Min-class Mahalanobis Distance"); ax4.set_ylabel("Count")
ax4.legend(); ax4.grid(alpha=0.3)
ood_dist_path = OUTPUT_DIR / "ood_score_distribution.png"
plt.savefig(ood_dist_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ood_dist_path}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 13: OOD Inference Helper (full pipeline)
# ─────────────────────────────────────────────

def predict_with_ood(image_path: str,
                     yolo_model,
                     feat_session: ort.InferenceSession,
                     threshold: float,
                     img_size: int = 640,
                     conf: float = 0.25,
                     iou: float = 0.45,
                     device=0) -> dict:
    """
    Run YOLOv8 detection + Mahalanobis OOD check.
    Returns a dict with detections and OOD flag.
    """
    # Detection
    res    = yolo_model.predict(image_path, conf=conf, iou=iou,
                                verbose=False, device=device)[0]
    boxes  = res.boxes.xyxy.cpu().numpy()  if res.boxes else np.array([])
    scores = res.boxes.conf.cpu().numpy()  if res.boxes else np.array([])
    cls_ids= res.boxes.cls.cpu().numpy().astype(int) if res.boxes else np.array([])

    # Feature extraction for OOD
    img = cv2.imread(image_path)
    img = cv2.resize(img, (img_size, img_size))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tensor = (img.astype(np.float32) / 255.0).transpose(2, 0, 1)[None]
    feat   = feat_session.run(None, {"images": tensor})[0][0]
    maha   = min_class_mahalanobis(feat)
    is_ood = maha > threshold

    detections = []
    for box, score, cid in zip(boxes, scores, cls_ids):
        detections.append({
            "class_id"  : int(cid),
            "class_name": new_id2name.get(int(cid), "unknown"),
            "confidence": float(score),
            "bbox_xyxy" : box.tolist(),
        })

    return {
        "image_path"     : image_path,
        "detections"     : detections,
        "mahalanobis_dist": maha,
        "ood_threshold"  : threshold,
        "is_ood"         : bool(is_ood),
    }


# Quick demo on a few test images
print("\n── OOD Demo on Test Images ──────────────────────")
test_imgs = sorted((FILTERED_DIR / "test" / "images").iterdir())[:5]
for ip in test_imgs:
    out = predict_with_ood(str(ip), best_model, feat_sess, ood_threshold,
                           img_size=IMG_SIZE, device=DEVICE)
    flag = "🚨 OOD" if out["is_ood"] else "✅ In-dist"
    print(f"  {ip.name:40s}  Maha={out['mahalanobis_dist']:7.2f}  {flag}  "
          f"  dets={len(out['detections'])}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 14: Save label.json
# ─────────────────────────────────────────────

label_json = {
    "classes"   : new_classes,
    "id2name"   : {str(k): v for k, v in new_id2name.items()},
    "name2id"   : new_name2id,
    "num_classes": len(new_classes),
}
label_path = OUTPUT_DIR / "label.json"
with open(label_path, "w") as f:
    json.dump(label_json, f, indent=2)
print(f"Saved: {label_path}")
print(json.dumps(label_json, indent=2))

In [ ]:
# ─────────────────────────────────────────────
# CELL 15: Save ood_config.json
# ─────────────────────────────────────────────

# Serialise per-class means (precision matrices are large; store path reference)
ood_config = {
    "method"           : "mahalanobis",
    "feature_extractor": "feature_extractor.onnx",
    "img_size"         : IMG_SIZE,
    "threshold"        : ood_threshold,
    "threshold_rule"   : "val_mean_plus_3std",
    "val_mean"         : float(val_scores.mean()),
    "val_std"          : float(val_scores.std()),
    "num_val_samples"  : len(val_scores),
    "classes"          : new_classes,
    "class_means"      : {str(k): v.tolist() for k, v in class_means.items()},
    # precision matrices stored separately (NumPy) to keep JSON readable
    "class_precisions_file": "ood_precisions.npz",
    "global_mean"      : global_mu.tolist(),
}

ood_config_path = OUTPUT_DIR / "ood_config.json"
with open(ood_config_path, "w") as f:
    json.dump(ood_config, f, indent=2)
print(f"Saved: {ood_config_path}")

# Save precision matrices as NPZ
prec_data = {f"class_{k}": v for k, v in class_covs.items()}
prec_data["global"] = global_prec
np.savez_compressed(OUTPUT_DIR / "ood_precisions.npz", **prec_data)
print(f"Saved: {OUTPUT_DIR / 'ood_precisions.npz'}")

In [ ]:
# ─────────────────────────────────────────────
# CELL 16: Final Summary
# ─────────────────────────────────────────────

print("\n" + "=" * 60)
print("ALL OUTPUTS SAVED TO:", OUTPUT_DIR)
print("=" * 60)
for p in sorted(OUTPUT_DIR.iterdir()):
    size = p.stat().st_size / 1024
    print(f"  {p.name:40s}  {size:8.1f} KB")

print("\n── Files to download for deployment ──────────────")
deploy_files = ["model.onnx", "feature_extractor.onnx",
                "label.json", "ood_config.json", "ood_precisions.npz"]
for fn in deploy_files:
    fp = OUTPUT_DIR / fn
    if fp.exists():
        print(f"  ✅  {fn}")
    else:
        print(f"  ❌  {fn}  (NOT FOUND)")